# 💧 Classificação de Potabilidade da Água com Machine Learning

---

## Contexto do Problema

O acesso à água potável é um direito humano fundamental e um dos maiores desafios de saúde pública no mundo. Segundo a OMS, aproximadamente **2 bilhões de pessoas** não têm acesso à água segura para consumo, o que causa doenças e mortes evitáveis.

A verificação manual da qualidade da água exige análises laboratoriais custosas e demoradas. O uso de **Machine Learning** pode acelerar esse processo, permitindo que sistemas automatizados classifiquem a potabilidade da água a partir de parâmetros físico-químicos mensuráveis.

### Dataset Utilizado
O dataset **Water Potability** contém medições de qualidade da água de 3.276 corpos d'água diferentes, com as seguintes variáveis:

| Feature | Descrição | Unidade |
|---|---|---|
| `ph` | Potencial hidrogeniônico | 0-14 |
| `Hardness` | Dureza (cálcio e magnésio) | mg/L |
| `Solids` | Sólidos totais dissolvidos (TDS) | ppm |
| `Chloramines` | Concentração de cloraminas | ppm |
| `Sulfate` | Concentração de sulfatos | mg/L |
| `Conductivity` | Condutividade elétrica | μS/cm |
| `Organic_carbon` | Carbono orgânico total (TOC) | ppm |
| `Trihalomethanes` | Concentração de trihalometanos | μg/L |
| `Turbidity` | Turbidez da água | NTU |
| `Potability` | **Variável alvo**: 1 = Potável, 0 = Não Potável | — |

**Fonte:** [Kaggle – Water Quality](https://www.kaggle.com/datasets/adityakadiwal/water-potability)

---

### Objetivo
Treinar e comparar modelos de classificação (KNN, Árvore de Decisão, Naive Bayes e SVM) para prever se uma amostra de água é segura para consumo humano, seguindo as melhores práticas de Engenharia de Sistemas de Software Inteligentes.

## 1. Instalação e Importação de Dependências

A seguir, importamos todas as bibliotecas necessárias para o projeto. O `scikit-learn` é a principal biblioteca de ML utilizada, enquanto `pandas` e `numpy` lidam com a manipulação dos dados e `matplotlib`/`seaborn` são responsáveis pelas visualizações.

In [ ]:
# Instalação de bibliotecas adicionais (caso necessário no Colab)
# !pip install scikit-learn pandas numpy matplotlib seaborn joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

# Scikit-Learn: Pré-processamento
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Scikit-Learn: Algoritmos de Classificação
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Scikit-Learn: Métricas de Avaliação
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score, roc_auc_score, roc_curve
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

print('✅ Bibliotecas importadas com sucesso!')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   sklearn : {__import__("sklearn").__version__}')

## 2. Carga dos Dados

O dataset é carregado diretamente via URL do GitHub, sem necessidade de download ou configuração local. Isso garante reprodutibilidade total do notebook.

In [ ]:
# URL do dataset no GitHub (versão raw para leitura direta)
URL_DATASET = "https://raw.githubusercontent.com/MainakRepositor/Datasets/master/water_potability.csv"

print(f'⬇️  Carregando dataset a partir de:\n   {URL_DATASET}\n')

df = pd.read_csv(URL_DATASET)

print(f'✅ Dataset carregado com sucesso!')
print(f'   Dimensões : {df.shape[0]} linhas × {df.shape[1]} colunas')
print(f'   Memória   : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

df.head(10)

## 3. Análise Exploratória dos Dados (EDA)

Antes de modelar, é fundamental entender a estrutura e as características dos dados. A análise exploratória nos permite identificar problemas como valores ausentes, distribuições assimétricas e desbalanceamento de classes — fatores que impactam diretamente o desempenho dos modelos.

In [ ]:
# Informações gerais do dataset
print('📋 INFORMAÇÕES DO DATASET')
print('=' * 50)
df.info()

print('\n📊 ESTATÍSTICAS DESCRITIVAS')
print('=' * 50)
df.describe().round(3)

In [ ]:
# Verificação de valores ausentes
print('🔍 VALORES AUSENTES POR COLUNA')
print('=' * 50)

missing_data = pd.DataFrame({
    'Contagem': df.isnull().sum(),
    'Percentual (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_data = missing_data[missing_data['Contagem'] > 0]

if missing_data.empty:
    print('Nenhum valor ausente encontrado.')
else:
    print(missing_data)
    print('\n⚠️  Atenção: Valores ausentes serão tratados na etapa de pré-processamento.')

In [ ]:
# Distribuição da variável alvo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

contagem = df['Potability'].value_counts()
labels = ['Não Potável (0)', 'Potável (1)']
cores = ['#e74c3c', '#2ecc71']

# Gráfico de barras
bars = axes[0].bar(labels, contagem.values, color=cores, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, contagem.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Distribuição da Classe Alvo (Potability)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Contagem')
axes[0].set_ylim(0, max(contagem.values) * 1.2)

# Gráfico de pizza
axes[1].pie(contagem.values, labels=labels, colors=cores, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporção das Classes', fontsize=13, fontweight='bold')

plt.suptitle('Análise da Variável Alvo: Potabilidade da Água', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f'\n📌 Razão de desbalanceamento: {contagem[0]/contagem[1]:.2f}:1 (Não Potável:Potável)')
print('   O dataset apresenta leve desbalanceamento — usaremos stratify na divisão treino/teste.')

In [ ]:
# Distribuição das features
features = df.columns.drop('Potability').tolist()

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()

for idx, feature in enumerate(features):
    for potability, cor, label in [(0, '#e74c3c', 'Não Potável'), (1, '#2ecc71', 'Potável')]:
        subset = df[df['Potability'] == potability][feature].dropna()
        axes[idx].hist(subset, bins=30, alpha=0.6, color=cor, label=label, edgecolor='white')
    axes[idx].set_title(feature, fontweight='bold')
    axes[idx].legend(fontsize=8)
    axes[idx].set_ylabel('Frequência')

plt.suptitle('Distribuição das Features por Classe', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlação
plt.figure(figsize=(11, 8))
corr_matrix = df.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, cbar_kws={'label': 'Correlação de Pearson'}
)
plt.title('Matriz de Correlação entre as Variáveis', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print('📌 Observação: As features apresentam baixa correlação entre si,')
print('   o que é favorável para algoritmos que assumem independência (ex: Naive Bayes).')

In [ ]:
# Boxplots para identificar outliers
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()

for idx, feature in enumerate(features):
    df.boxplot(column=feature, by='Potability', ax=axes[idx],
               boxprops=dict(color='#3498db'),
               medianprops=dict(color='#e74c3c', linewidth=2))
    axes[idx].set_title(feature, fontweight='bold')
    axes[idx].set_xlabel('Potability (0=Não, 1=Sim)')
    axes[idx].set_ylabel(feature)

plt.suptitle('Boxplots: Distribuição das Features por Classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Pré-processamento: Separação Treino/Teste (Holdout)

Separamos os dados em conjunto de **treino (80%)** e **teste (20%)**, usando `stratify=y` para garantir que a proporção das classes seja mantida em ambos os conjuntos. Essa estratégia, chamada de **holdout estratificado**, evita que o modelo seja avaliado em uma distribuição diferente da treinada.

⚠️ **Importante:** O conjunto de teste só será usado na avaliação final. Todas as etapas de desenvolvimento (inclusive a otimização de hiperparâmetros) usarão apenas o conjunto de treino, com Cross-Validation.

In [ ]:
# Separação entre features (X) e variável alvo (y)
X = df.drop('Potability', axis=1)
y = df['Potability']

# Holdout estratificado: 80% treino, 20% teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y  # Mantém a proporção das classes
)

print('📂 DIVISÃO TREINO / TESTE (HOLDOUT)')
print('=' * 50)
print(f'   Total de amostras   : {len(df):>5}')
print(f'   Amostras de treino  : {len(X_train):>5} ({len(X_train)/len(df)*100:.0f}%)')
print(f'   Amostras de teste   : {len(X_test):>5} ({len(X_test)/len(df)*100:.0f}%)')
print()
print('   Distribuição das classes:')
print(f'     Treino  → Não Potável: {y_train.value_counts()[0]} | Potável: {y_train.value_counts()[1]}')
print(f'     Teste   → Não Potável: {y_test.value_counts()[0]}  | Potável: {y_test.value_counts()[1]}')
print()
print(f'   Features             : {X.shape[1]}')
print(f'   Features: {list(X.columns)}')

## 5. Transformação de Dados e Construção dos Pipelines

Nesta etapa, construímos **Pipelines** que integram o pré-processamento e a modelagem em uma única sequência encadeada. Isso garante que:

1. **Sem vazamento de dados (data leakage):** Os parâmetros de transformação (média, desvio padrão, min/max) são calculados somente sobre o conjunto de treino e aplicados ao teste.
2. **Reprodutibilidade:** O pipeline encapsula todo o processo, do dado bruto à predição.
3. **Facilidade de implantação:** O modelo exportado já inclui os transformadores, aceitando dados brutos.

### Estratégia de Pré-processamento:
- **Imputação:** Valores ausentes substituídos pela **mediana** da coluna (robusta a outliers)
- **KNN, Árvore de Decisão, SVM → Padronização** (`StandardScaler`: média=0, dp=1)
- **Naive Bayes → Normalização** (`MinMaxScaler`: escala [0,1]), pois o GaussianNB assume distribuições, não requer distâncias euclidianas

> **Nota:** A Árvore de Decisão não é sensível à escala dos dados, mas incluímos a padronização no pipeline por consistência e para facilitar futuras comparações.

In [ ]:
# Estratégia de Cross-Validation: StratifiedKFold com 5 folds
# Mantém a proporção de classes em cada fold — essencial para datasets desbalanceados
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Preprocessador 1: Imputação mediana + Padronização (para KNN, DT, SVM)
preprocessor_std = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Trata valores ausentes
    ('scaler', StandardScaler())                    # Padronização: z = (x - μ) / σ
])

# Preprocessador 2: Imputação mediana + Normalização (para Naive Bayes)
preprocessor_norm = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Trata valores ausentes
    ('scaler', MinMaxScaler())                      # Normalização: x' = (x - min) / (max - min)
])

print('✅ Preprocessadores definidos:')
print('   📊 preprocessor_std  → Imputação (mediana) + StandardScaler')
print('   📊 preprocessor_norm → Imputação (mediana) + MinMaxScaler')
print(f'\n   Cross-Validation: StratifiedKFold (n_splits=5, shuffle=True, seed=42)')
print(f'   Métrica de otimização: F1-Score (adequada para classes ligeiramente desbalanceadas)')

## 6. Modelagem e Otimização de Hiperparâmetros

Para cada algoritmo, construímos um Pipeline completo e aplicamos **GridSearchCV** para encontrar os melhores hiperparâmetros. O GridSearchCV testa todas as combinações definidas usando Cross-Validation, garantindo que a escolha dos hiperparâmetros não seja influenciada pelo conjunto de teste.

> **Métrica de otimização: F1-Score** — escolhida por ser mais informativa que a acurácia em datasets com algum desbalanceamento.

In [ ]:
# ================================================================
# MODELO 1: K-Nearest Neighbors (KNN)
# ================================================================
print('🔧 Treinando KNN com GridSearchCV...')
print('   Sensível à escala → usando StandardScaler')

knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_std),
    ('classifier', KNeighborsClassifier())
])

knn_params = {
    'classifier__n_neighbors': [3, 5, 7, 9, 11, 15],  # Número de vizinhos
    'classifier__weights'    : ['uniform', 'distance'], # Pesos dos vizinhos
    'classifier__metric'     : ['euclidean', 'manhattan'] # Métrica de distância
}

knn_grid = GridSearchCV(
    knn_pipeline, knn_params,
    cv=cv_strategy, scoring='f1',
    n_jobs=-1, verbose=0, refit=True
)
knn_grid.fit(X_train, y_train)

print(f'\n   ✅ Concluído!')
print(f'   Melhores parâmetros : {knn_grid.best_params_}')
print(f'   Melhor F1 (CV)      : {knn_grid.best_score_:.4f}')

In [ ]:
# ================================================================
# MODELO 2: Árvore de Classificação (Decision Tree)
# ================================================================
print('🔧 Treinando Decision Tree com GridSearchCV...')
print('   Invariante à escala, mas padronizamos por consistência')

dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_std),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

dt_params = {
    'classifier__criterion'       : ['gini', 'entropy'],      # Critério de divisão
    'classifier__max_depth'       : [3, 5, 7, 10, None],      # Profundidade máxima
    'classifier__min_samples_split': [2, 5, 10, 20],          # Amostras mínimas para dividir
    'classifier__min_samples_leaf': [1, 2, 5]                 # Amostras mínimas por folha
}

dt_grid = GridSearchCV(
    dt_pipeline, dt_params,
    cv=cv_strategy, scoring='f1',
    n_jobs=-1, verbose=0, refit=True
)
dt_grid.fit(X_train, y_train)

print(f'\n   ✅ Concluído!')
print(f'   Melhores parâmetros : {dt_grid.best_params_}')
print(f'   Melhor F1 (CV)      : {dt_grid.best_score_:.4f}')

In [ ]:
# ================================================================
# MODELO 3: Naive Bayes (GaussianNB)
# ================================================================
print('🔧 Treinando Naive Bayes com GridSearchCV...')
print('   Assume distribuição gaussiana → usando MinMaxScaler')

nb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_norm),
    ('classifier', GaussianNB())
])

nb_params = {
    'classifier__var_smoothing': [1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}

nb_grid = GridSearchCV(
    nb_pipeline, nb_params,
    cv=cv_strategy, scoring='f1',
    n_jobs=-1, verbose=0, refit=True
)
nb_grid.fit(X_train, y_train)

print(f'\n   ✅ Concluído!')
print(f'   Melhores parâmetros : {nb_grid.best_params_}')
print(f'   Melhor F1 (CV)      : {nb_grid.best_score_:.4f}')

In [ ]:
# ================================================================
# MODELO 4: Support Vector Machine (SVM)
# ================================================================
print('🔧 Treinando SVM com GridSearchCV...')
print('   Muito sensível à escala → usando StandardScaler')
print('   ⏳ Este pode demorar alguns minutos...')

svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_std),
    ('classifier', SVC(probability=True, random_state=42))
])

svm_params = {
    'classifier__C'     : [0.1, 1, 10, 100],             # Parâmetro de regularização
    'classifier__kernel': ['rbf', 'linear', 'poly'],      # Tipo de kernel
    'classifier__gamma' : ['scale', 'auto']               # Coeficiente do kernel
}

svm_grid = GridSearchCV(
    svm_pipeline, svm_params,
    cv=cv_strategy, scoring='f1',
    n_jobs=-1, verbose=0, refit=True
)
svm_grid.fit(X_train, y_train)

print(f'\n   ✅ Concluído!')
print(f'   Melhores parâmetros : {svm_grid.best_params_}')
print(f'   Melhor F1 (CV)      : {svm_grid.best_score_:.4f}')

## 7. Avaliação e Comparação dos Modelos

Agora avaliamos os modelos no **conjunto de teste** (dados nunca vistos durante o treinamento). Utilizamos múltiplas métricas para uma avaliação completa:

- **Acurácia:** Proporção geral de acertos
- **F1-Score:** Média harmônica entre Precisão e Recall (equilibra falsos positivos e negativos)
- **ROC-AUC:** Capacidade discriminativa do modelo em diferentes limiares de classificação
- **Matriz de Confusão:** Detalha os erros e acertos por classe

In [ ]:
# Dicionário com os melhores modelos de cada algoritmo
modelos = {
    'KNN'                : knn_grid.best_estimator_,
    'Árvore de Decisão'  : dt_grid.best_estimator_,
    'Naive Bayes'        : nb_grid.best_estimator_,
    'SVM'                : svm_grid.best_estimator_
}

resultados = []

print('📋 RELATÓRIOS DE CLASSIFICAÇÃO — CONJUNTO DE TESTE')
print('=' * 65)

for nome, modelo in modelos.items():
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, zero_division=0)
    roc = roc_auc_score(y_test, y_prob)

    resultados.append({
        'Modelo'   : nome,
        'Acurácia' : round(acc, 4),
        'F1-Score' : round(f1, 4),
        'ROC-AUC'  : round(roc, 4)
    })

    print(f'\n🤖 Modelo: {nome}')
    print('-' * 55)
    print(classification_report(
        y_test, y_pred,
        target_names=['Não Potável (0)', 'Potável (1)'],
        zero_division=0
    ))

df_resultados = pd.DataFrame(resultados)
print('\n📊 TABELA RESUMO DOS RESULTADOS')
print('=' * 55)
print(df_resultados.to_string(index=False))

In [ ]:
# Matrizes de Confusão para todos os modelos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, (nome, modelo) in enumerate(modelos.items()):
    y_pred = modelo.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Não Potável', 'Potável']
    )
    disp.plot(ax=axes[idx], colorbar=False, cmap='Blues')
    axes[idx].set_title(f'{nome}', fontsize=13, fontweight='bold', pad=10)

    # Anotação de métricas no gráfico
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    axes[idx].set_xlabel(
        f'Predito\n(Acurácia={acc:.3f} | F1={f1:.3f})',
        fontsize=10
    )

plt.suptitle('Matrizes de Confusão — Conjunto de Teste', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico comparativo de métricas
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metricas = ['Acurácia', 'F1-Score', 'ROC-AUC']
cores_metricas = ['#3498db', '#e74c3c', '#2ecc71']
x = np.arange(len(df_resultados))
width = 0.25

# Gráfico de barras agrupadas
for i, (metrica, cor) in enumerate(zip(metricas, cores_metricas)):
    offset = (i - 1) * width
    bars = axes[0].bar(x + offset, df_resultados[metrica], width, label=metrica,
                        color=cor, alpha=0.85, edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        axes[0].annotate(f'{h:.3f}',
                         xy=(bar.get_x() + bar.get_width()/2, h),
                         xytext=(0, 3), textcoords='offset points',
                         ha='center', va='bottom', fontsize=8)

axes[0].set_xlabel('Modelo', fontsize=12)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Comparação de Métricas por Modelo', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_resultados['Modelo'], fontsize=10)
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, 1.05)
axes[0].axhline(y=0.6, color='gray', linestyle='--', alpha=0.5, label='Linha de referência 60%')

# Curvas ROC
for nome, modelo in modelos.items():
    y_prob = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[1].plot(fpr, tpr, linewidth=2, label=f'{nome} (AUC={auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.500)')
axes[1].set_xlabel('Taxa de Falso Positivo (FPR)', fontsize=12)
axes[1].set_ylabel('Taxa de Verdadeiro Positivo (TPR)', fontsize=12)
axes[1].set_title('Curvas ROC', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation final para confirmação dos resultados
print('🔄 VALIDAÇÃO CRUZADA FINAL (5-Fold Stratified)')
print('=' * 65)
print('   (Aplicado sobre todo o dataset de treino, usando o melhor estimador)')
print()

for nome, modelo in modelos.items():
    cv_scores = cross_val_score(modelo, X_train, y_train,
                                cv=cv_strategy, scoring='f1', n_jobs=-1)
    print(f'   {nome:<22} | F1 por fold: {[f"{s:.3f}" for s in cv_scores]}')
    print(f'   {"":22} | Média: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print()

## 8. Seleção e Exportação do Melhor Modelo

Selecionamos o modelo com melhor F1-Score no conjunto de teste e o exportamos usando `joblib`. O arquivo `.pkl` resultante encapsula **todo o pipeline** (imputação + normalização/padronização + classificador), podendo ser carregado e usado diretamente para novas predições sem necessidade de aplicar transformações manualmente.

In [ ]:
# Identificação do melhor modelo pelo F1-Score no conjunto de teste
idx_melhor = df_resultados['F1-Score'].idxmax()
melhor_nome   = df_resultados.loc[idx_melhor, 'Modelo']
melhor_modelo = modelos[melhor_nome]
melhor_f1     = df_resultados.loc[idx_melhor, 'F1-Score']
melhor_acc    = df_resultados.loc[idx_melhor, 'Acurácia']
melhor_roc    = df_resultados.loc[idx_melhor, 'ROC-AUC']

print('🏆 MELHOR MODELO SELECIONADO')
print('=' * 50)
print(f'   Algoritmo : {melhor_nome}')
print(f'   Acurácia  : {melhor_acc:.4f}')
print(f'   F1-Score  : {melhor_f1:.4f}')
print(f'   ROC-AUC   : {melhor_roc:.4f}')
print()
print('   Pipeline do modelo selecionado:')
print(melhor_modelo)

In [ ]:
# Exportação do modelo com joblib
NOME_ARQUIVO = 'modelo_potabilidade.pkl'
joblib.dump(melhor_modelo, NOME_ARQUIVO)
print(f'💾 Modelo exportado: "{NOME_ARQUIVO}"')

# Verificação: carregando o modelo e testando
modelo_carregado = joblib.load(NOME_ARQUIVO)
y_pred_check = modelo_carregado.predict(X_test)
f1_check = f1_score(y_test, y_pred_check, zero_division=0)

print(f'\n✅ Verificação após carregamento:')
print(f'   F1-Score original   : {melhor_f1:.4f}')
print(f'   F1-Score verificado : {f1_check:.4f}')
assert abs(melhor_f1 - f1_check) < 1e-6, '❌ ERRO: F1-Score diverge após carregamento!'
print('   Integridade do modelo confirmada ✅')

In [ ]:
# Exemplo de predição com o modelo exportado
print('🔮 EXEMPLO DE PREDIÇÃO COM O MODELO EXPORTADO')
print('=' * 55)

# Amostra de dados brutos (com valores ausentes como NaN para testar a imputação)
amostra_dict = {
    'ph'              : 7.08,
    'Hardness'        : 204.89,
    'Solids'          : 20791.32,
    'Chloramines'     : 7.30,
    'Sulfate'         : np.nan,   # Valor ausente — será imputado pelo pipeline
    'Conductivity'    : 564.31,
    'Organic_carbon'  : 10.38,
    'Trihalomethanes' : 86.99,
    'Turbidity'       : 2.96
}

amostra_df = pd.DataFrame([amostra_dict])
print('   Dados de entrada (brutos):')
print(amostra_df.to_string(index=False))

predicao = modelo_carregado.predict(amostra_df)[0]
probabilidades = modelo_carregado.predict_proba(amostra_df)[0]

print(f'\n   Resultado da predição:')
print(f'   Classe predita  : {predicao} ({"✅ POTÁVEL" if predicao == 1 else "❌ NÃO POTÁVEL"})')
print(f'   Prob. Não Potável : {probabilidades[0]:.4f} ({probabilidades[0]*100:.1f}%)')
print(f'   Prob. Potável     : {probabilidades[1]:.4f} ({probabilidades[1]*100:.1f}%)')

## 9. Conclusões

Neste notebook, desenvolvemos um pipeline completo de Machine Learning para classificação de potabilidade da água, seguindo as boas práticas de Engenharia de Sistemas de Software Inteligentes:

### Resultados
Os modelos foram treinados e avaliados com as seguintes métricas no conjunto de teste:

| Etapa | Técnica Utilizada |
|---|---|
| Carga dos dados | URL direta do GitHub |
| Divisão treino/teste | Holdout Estratificado (80/20) |
| Valores ausentes | SimpleImputer (mediana) |
| Transformação de dados | StandardScaler + MinMaxScaler |
| Encapsulamento | sklearn Pipeline |
| Algoritmos | KNN, Árvore de Decisão, Naive Bayes, SVM |
| Otimização | GridSearchCV |
| Validação | StratifiedKFold (5 folds) |
| Exportação | joblib (.pkl) |

### Observações
- O dataset de potabilidade da água apresenta **desbalanceamento moderado** (~61% Não Potável vs ~39% Potável), o que justifica o uso de F1-Score como métrica principal
- As features apresentam **baixa correlação** entre si, o que beneficia o Naive Bayes
- O **SVM com kernel RBF** costuma ser mais poderoso em datasets de média dimensão, mas requer mais tempo de treinamento
- O melhor modelo foi exportado e pode ser integrado à aplicação web descrita na parte 4 do trabalho